# Strands Evals — Trace-Level and Session-Level Evaluators

## Overview

This notebook demonstrates how to evaluate multi-turn conversations using [Strands Evals](https://strandsagents.com/docs/user-guide/evals-sdk/evaluators/) built-in evaluators. These evaluators use an LLM-as-judge approach to assess different dimensions of agent quality, from individual response coherence to full-session goal achievement.

Strands Evals evaluators operate at three levels:

| Level | Scope | Evaluators | What They Assess |
|-------|-------|------------|------------------|
| **TRACE_LEVEL** | Single turn (most recent) | `CoherenceEvaluator`, `FaithfulnessEvaluator`, `HelpfulnessEvaluator`, `ResponseRelevanceEvaluator` | Quality of individual responses in context |
| **SESSION_LEVEL** | Full conversation | `GoalSuccessRateEvaluator`, `InteractionsEvaluator` | End-to-end goal achievement and conversation patterns |
| **OUTPUT_LEVEL** | Single response | `OutputEvaluator` | Response quality against custom rubrics |

All trace-level and session-level evaluators require OpenTelemetry traces captured via `StrandsEvalsTelemetry` and mapped to `Session` objects via `StrandsInMemorySessionMapper`.

## What You'll Learn

1. How to use trace-level evaluators: `CoherenceEvaluator` (5-level), `FaithfulnessEvaluator` (5-level), `HelpfulnessEvaluator` (7-level), `ResponseRelevanceEvaluator` (5-level)
2. How to use session-level evaluators: `GoalSuccessRateEvaluator` (basic + assertion modes), `InteractionsEvaluator`
3. How to combine multiple evaluators in a single `Experiment` for multi-dimensional assessment
4. How to interpret and compare scores across evaluators to identify specific areas of weakness

## Scoring Scales

### Trace-Level Evaluators

| Evaluator | Levels | Score Range | What It Measures |
|-----------|--------|-------------|------------------|
| `CoherenceEvaluator` | Not At All → Completely Yes | 0.0, 0.25, 0.5, 0.75, 1.0 | Logical cohesion and reasoning quality |
| `FaithfulnessEvaluator` | Not At All → Completely Yes | 0.0, 0.25, 0.5, 0.75, 1.0 | Whether response is grounded in conversation history |
| `HelpfulnessEvaluator` | Not helpful → Above and beyond | 0.0, 0.167, 0.333, 0.5, 0.667, 0.833, 1.0 | Response utility from user perspective |
| `ResponseRelevanceEvaluator` | Not At All → Completely Yes | 0.0, 0.25, 0.5, 0.75, 1.0 | Relevance to the user's question |

### Session-Level Evaluators

| Evaluator | Mode | Score | What It Measures |
|-----------|------|-------|------------------|
| `GoalSuccessRateEvaluator` | Basic | Yes=1.0, No=0.0 | Were user goals achieved? (inferred from conversation) |
| `GoalSuccessRateEvaluator` | Assertion | SUCCESS=1.0, FAILURE=0.0 | Were explicit success criteria met? (human-authored assertions) |
| `InteractionsEvaluator` | Custom rubric | 0.0–1.0 | Conversation patterns and interaction quality |

## 1. Setup

Recreate the travel booking agent and set up telemetry for trace capture.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
from strands import Agent, tool
from datetime import date

bookings: dict = {}
booking_counter = 0

@tool
def search_flights(origin: str, destination: str, departure_date: str, return_date: str = None) -> str:
    """Search for available flights between two cities.

    Args:
        origin: Departure city or airport code (e.g. 'New York' or 'JFK')
        destination: Arrival city or airport code (e.g. 'London' or 'LHR')
        departure_date: Departure date in YYYY-MM-DD format
        return_date: Optional return date in YYYY-MM-DD format for round trips
    """
    flights = [
        {"flight": "AA101", "airline": "American Airlines", "departs": "08:00", "arrives": "20:00", "price": 450, "class": "Economy"},
        {"flight": "BA202", "airline": "British Airways",   "departs": "11:30", "arrives": "23:45", "price": 620, "class": "Economy"},
        {"flight": "UA303", "airline": "United Airlines",   "departs": "14:00", "arrives": "02:15", "price": 890, "class": "Business"},
    ]
    trip = f"round-trip (return: {return_date})" if return_date else "one-way"
    lines = [f"Flights from {origin} to {destination} on {departure_date} ({trip}):"]
    for f in flights:
        lines.append(f"  {f['flight']} | {f['airline']} | {f['departs']}-{f['arrives']} | ${f['price']} | {f['class']}")
    return "\n".join(lines)

@tool
def search_hotels(city: str, check_in: str, check_out: str, guests: int = 1) -> str:
    """Search for available hotels in a city.

    Args:
        city: Destination city
        check_in: Check-in date in YYYY-MM-DD format
        check_out: Check-out date in YYYY-MM-DD format
        guests: Number of guests (default 1)
    """
    hotels = [
        {"name": "Grand Plaza Hotel",  "stars": 5, "price_per_night": 320, "amenities": "Pool, Spa, Restaurant"},
        {"name": "City Center Inn",     "stars": 3, "price_per_night": 95,  "amenities": "Free WiFi, Breakfast"},
        {"name": "Boutique Stay",       "stars": 4, "price_per_night": 180, "amenities": "Gym, Bar, Concierge"},
    ]
    nights = (date.fromisoformat(check_out) - date.fromisoformat(check_in)).days
    lines = [f"Hotels in {city} | {check_in} to {check_out} ({nights} nights, {guests} guest(s)):"]
    for h in hotels:
        total = h["price_per_night"] * nights
        lines.append(f"  {h['name']} ({h['stars']}*) | ${h['price_per_night']}/night (${total} total) | {h['amenities']}")
    return "\n".join(lines)

@tool
def book_flight(flight_number: str, passenger_name: str, origin: str, destination: str, travel_date: str) -> str:
    """Book a flight for a passenger.

    Args:
        flight_number: Flight number to book (e.g. 'AA101')
        passenger_name: Full name of the passenger
        origin: Departure city or airport
        destination: Arrival city or airport
        travel_date: Travel date in YYYY-MM-DD format
    """
    global booking_counter
    booking_counter += 1
    ref = f"FLT-{booking_counter:04d}"
    bookings[ref] = {"type": "flight", "flight": flight_number, "passenger": passenger_name,
                     "route": f"{origin} -> {destination}", "date": travel_date, "status": "Confirmed"}
    return f"Flight booked! Ref: {ref} | {flight_number} | {origin} -> {destination} on {travel_date} | Passenger: {passenger_name}"

@tool
def book_hotel(hotel_name: str, guest_name: str, city: str, check_in: str, check_out: str) -> str:
    """Book a hotel room for a guest.

    Args:
        hotel_name: Name of the hotel to book
        guest_name: Full name of the guest
        city: City where the hotel is located
        check_in: Check-in date in YYYY-MM-DD format
        check_out: Check-out date in YYYY-MM-DD format
    """
    global booking_counter
    booking_counter += 1
    ref = f"HTL-{booking_counter:04d}"
    bookings[ref] = {"type": "hotel", "hotel": hotel_name, "guest": guest_name,
                     "city": city, "check_in": check_in, "check_out": check_out, "status": "Confirmed"}
    return f"Hotel booked! Ref: {ref} | {hotel_name} in {city} | {check_in} to {check_out} | Guest: {guest_name}"

@tool
def get_bookings() -> str:
    """Retrieve all current bookings."""
    if not bookings:
        return "No bookings found."
    lines = ["Current bookings:"]
    for ref, b in bookings.items():
        if b["type"] == "flight":
            lines.append(f"  [{ref}] Flight {b['flight']} | {b['route']} on {b['date']} | {b['passenger']} | {b['status']}")
        else:
            lines.append(f"  [{ref}] Hotel {b['hotel']} in {b['city']} | {b['check_in']} to {b['check_out']} | {b['guest']} | {b['status']}")
    return "\n".join(lines)

@tool
def cancel_booking(booking_ref: str) -> str:
    """Cancel an existing booking.

    Args:
        booking_ref: Booking reference number (e.g. 'FLT-0001' or 'HTL-0002')
    """
    if booking_ref not in bookings:
        return f"Booking {booking_ref} not found."
    bookings[booking_ref]["status"] = "Cancelled"
    return f"Booking {booking_ref} has been cancelled."

ALL_TOOLS = [search_flights, search_hotels, book_flight, book_hotel, get_bookings, cancel_booking]

SYSTEM_PROMPT = (
    "You are a helpful travel booking assistant. You help users search for flights and hotels, "
    "make bookings, view existing reservations, and cancel bookings. "
    "Always confirm details with the user before completing a booking. "
    "Use today's date as context when dates are not fully specified."
)

print(f"Defined {len(ALL_TOOLS)} tools: {[t.__name__ for t in ALL_TOOLS]}")


In [ ]:
import nest_asyncio
nest_asyncio.apply()  # Required: Experiment.run_evaluations() uses asyncio.run() internally

from strands_evals import Case, Experiment, ActorSimulator
from strands_evals.evaluators import (
    CoherenceEvaluator,
    FaithfulnessEvaluator,
    HelpfulnessEvaluator,
    ResponseRelevanceEvaluator,
    GoalSuccessRateEvaluator,
)
from strands_evals.telemetry import StrandsEvalsTelemetry
from strands_evals.mappers import StrandsInMemorySessionMapper

# Setup telemetry for trace capture
telemetry = StrandsEvalsTelemetry().setup_in_memory_exporter()
memory_exporter = telemetry.in_memory_exporter

print('Evaluators and telemetry ready.')

## 2. Define Test Cases

We define test cases that exercise different aspects of the agent's multi-turn behavior. Each case includes:
- `input`: The initial user message
- `metadata["task_description"]`: The goal for `ActorSimulator` to track
- `expected_assertion`: Explicit success criteria for assertion-based `GoalSuccessRateEvaluator`

The `expected_assertion` field is key — it provides human-authored statements describing what the agent should have done, enabling more precise evaluation than basic goal inference.

In [ ]:
# Multi-step booking with explicit success assertions
case_booking = Case(
    name='multi-step-booking',
    input='I need flights from Seattle to Tokyo on November 10, 2025, and a hotel for 3 nights.',
    expected_assertion=(
        'The agent searched for flights from Seattle to Tokyo, presented options to the user, '
        'booked a flight after user confirmation, searched for hotels in Tokyo for Nov 10-13, '
        'and booked a hotel. The user received confirmed booking references for both.'
    ),
    metadata={
        'task_description': (
            'Search for flights from Seattle to Tokyo on Nov 10, compare options, book one, '
            'then search for hotels in Tokyo for 3 nights (Nov 10-13) and book one. '
            'End with confirmed booking references for both flight and hotel.'
        ),
        'category': 'multi_step',
    }
)

# Booking management with cancellation
case_manage = Case(
    name='review-and-cancel',
    input='Show me my current bookings. I think I need to cancel one of them.',
    expected_assertion=(
        'The agent retrieved all current bookings, presented them clearly, '
        'the user selected one to cancel, the agent cancelled it and confirmed '
        'the cancellation, then showed the remaining active bookings.'
    ),
    metadata={
        'task_description': (
            'Retrieve all bookings, let the user review them, cancel the one '
            'the user selects, and confirm the remaining bookings are intact.'
        ),
        'category': 'management',
    }
)

test_cases = [case_booking, case_manage]
print(f'Defined {len(test_cases)} test cases for evaluation.')

## 3. Task Function — Simulate and Capture Traces

The task function is the bridge between your test cases and the evaluators. It:
1. Creates an `ActorSimulator` to drive the conversation
2. Runs the multi-turn conversation with telemetry capture
3. Maps the captured spans into a `Session` object
4. Returns the output and trajectory (session) for evaluation

This is the standard pattern for combining simulation with evaluation in Strands Evals.

In [ ]:
def evaluation_task(case: Case) -> dict:
    """Run a simulated conversation and return output + session for evaluation.
    
    This function is passed to Experiment.run_evaluations() and called once per test case.
    It must return a dict with 'output' (str) and 'trajectory' (Session) keys.
    """
    # Reset booking state
    global bookings, booking_counter
    bookings = {}
    booking_counter = 0
    
    # Create simulator from test case
    simulator = ActorSimulator.from_case_for_user_simulator(
        case=case,
        max_turns=8
    )
    
    # Create agent with telemetry tracing
    agent = Agent(
        system_prompt=SYSTEM_PROMPT,
        tools=ALL_TOOLS,
        trace_attributes={
            'gen_ai.conversation.id': case.session_id,
            'session.id': case.session_id
        },
        callback_handler=None,
    )
    
    # Run conversation and collect spans
    all_spans = []
    user_message = case.input
    agent_text = ''
    
    while simulator.has_next():
        memory_exporter.clear()
        agent_response = agent(user_message)
        agent_text = str(agent_response)
        
        turn_spans = list(memory_exporter.get_finished_spans())
        all_spans.extend(turn_spans)
        
        user_result = simulator.act(agent_text)
        user_message = str(user_result.structured_output.message)
    
    # Map spans to Session for evaluators
    mapper = StrandsInMemorySessionMapper()
    session = mapper.map_to_session(all_spans, session_id=case.session_id)
    
    return {'output': agent_text, 'trajectory': session}

print('Task function ready.')

## 4. Section 6a — Trace-Level Evaluators (Per-Turn Analysis)

Trace-level evaluators assess the **most recent turn** in the conversation. They answer: "Given the full conversation history, was the agent's last response coherent, faithful, helpful, and relevant?"

These evaluators are most useful for:
- Pinpointing exactly where in a conversation quality degrades
- Comparing response quality across different agent configurations
- Catching surface-level defects (incoherence, hallucination, irrelevance)

We run all four trace-level evaluators together to get a multi-dimensional view of the agent's last response.

In [ ]:
# Run trace-level evaluators on the multi-step booking case
trace_evaluators = [
    CoherenceEvaluator(),         # 5-level: Not At All → Completely Yes
    FaithfulnessEvaluator(),      # 5-level: Not At All → Completely Yes
    HelpfulnessEvaluator(),       # 7-level: Not helpful → Above and beyond
    ResponseRelevanceEvaluator(), # 5-level: Not At All → Completely Yes
]

trace_experiment = Experiment(
    cases=[case_booking],
    evaluators=trace_evaluators
)

print('Running trace-level evaluators (assessing the last turn in each conversation)...')
print('This involves: ActorSimulator (user sim) + Agent (responses) + 4 LLM judges')
print()

trace_reports = trace_experiment.run_evaluations(evaluation_task)

# Reports are one per evaluator, each containing scores for all cases
for report in trace_reports:
    print(f'{report.evaluator_name:30s} | Overall: {report.overall_score:.3f}')
    for i, (score, passed, reason) in enumerate(zip(report.scores, report.test_passes, report.reasons)):
        case_name = report.cases[i].get('name', f'Case {i}')
        print(f'  {case_name}: Score={score:.3f} | Pass={passed} | {reason[:80]}...')

## 5. Section 6b — Session-Level Evaluators (Full Conversation Analysis)

Session-level evaluators assess the **entire conversation** as a unit. They answer: "Did the agent accomplish the user's goal across the full session?"

### GoalSuccessRateEvaluator — Two Modes

| Mode | How It Works | When to Use |
|------|-------------|-------------|
| **Basic** | The judge LLM infers user goals from the conversation and checks whether they were met. Returns Yes/No. | When you don't have explicit success criteria — good for exploratory evaluation |
| **Assertion** | The judge evaluates against `expected_assertion` — human-authored statements describing expected agent behavior. Returns SUCCESS/FAILURE. | When you have specific, testable success criteria — better for regression testing |

Assertion mode is more precise because it judges against explicit criteria you define upfront, rather than relying on the judge to infer what the user wanted.

In [ ]:
# Run GoalSuccessRateEvaluator in both modes
# Basic mode: infers goals from conversation
# Assertion mode: evaluates against expected_assertion field on the Case

session_evaluators = [
    GoalSuccessRateEvaluator(),  # Uses assertion mode automatically when expected_assertion is present
]

session_experiment = Experiment(
    cases=test_cases,  # Both cases have expected_assertion set
    evaluators=session_evaluators
)

print('Running session-level evaluators (assessing full conversations)...')
print()

session_reports = session_experiment.run_evaluations(evaluation_task)

for report in session_reports:
    print(f'Evaluator: {report.evaluator_name} | Overall: {report.overall_score:.1f}')
    for i, (score, passed, reason) in enumerate(zip(report.scores, report.test_passes, report.reasons)):
        case_name = report.cases[i].get('name', f'Case {i}')
        print(f'  {case_name}: Score={score:.1f} | Pass={passed}')
        print(f'    Reasoning: {reason[:150]}...')

## 6. Section 6c — Combining Multiple Evaluators

The real power of Strands Evals comes from combining trace-level and session-level evaluators in a single `Experiment`. This gives you a multi-dimensional view:

- **Trace-level scores** tell you about the quality of individual responses
- **Session-level scores** tell you whether the overall goal was achieved

A conversation where every turn scores well on coherence and helpfulness but fails on goal success reveals a specific pattern: the agent is producing good individual responses but failing to drive the conversation toward completion. Conversely, a conversation that achieves the goal but has low coherence scores suggests the agent is effective but rough around the edges.

In [ ]:
# Combine trace-level + session-level evaluators in one experiment
combined_evaluators = [
    CoherenceEvaluator(),
    HelpfulnessEvaluator(),
    GoalSuccessRateEvaluator(),
]

combined_experiment = Experiment(
    cases=test_cases,
    evaluators=combined_evaluators
)

print('Running combined evaluation (trace + session level)...')
print()

combined_reports = combined_experiment.run_evaluations(evaluation_task)

# Build a per-case summary from per-evaluator reports
case_scores = {}  # case_name -> {evaluator_name: score}
for report in combined_reports:
    for i, case_dict in enumerate(report.cases):
        name = case_dict.get('name', f'Case {i}')
        if name not in case_scores:
            case_scores[name] = {}
        case_scores[name][report.evaluator_name] = report.scores[i]

print(f'{"Case":25s} | {"Coherence":>10s} | {"Helpfulness":>12s} | {"Goal Success":>13s}')
print('-' * 70)
for case_name, scores in case_scores.items():
    coh = scores.get('CoherenceEvaluator', -1)
    hlp = scores.get('HelpfulnessEvaluator', -1)
    goal = scores.get('GoalSuccessRateEvaluator', -1)
    print(f'{case_name:25s} | {coh:10.3f} | {hlp:12.3f} | {goal:13.1f}')

print()
print('Per-evaluator overall scores:')
for report in combined_reports:
    print(f'  {report.evaluator_name:30s} | {report.overall_score:.3f}')

### Interpreting Multi-Dimensional Results

| Pattern | Coherence | Helpfulness | Goal Success | Diagnosis |
|---------|-----------|-------------|--------------|----------|
| Strong across the board | ≥0.75 | ≥0.667 | 1.0 | Agent is performing well |
| Good turns, failed goal | ≥0.75 | ≥0.667 | 0.0 | Agent produces good responses but doesn't drive toward completion |
| Achieved goal, rough turns | <0.5 | <0.5 | 1.0 | Agent is effective but incoherent — may frustrate users |
| Weak across the board | <0.5 | <0.5 | 0.0 | Fundamental issues with the agent's prompt, tools, or model |

## Next Steps

- Continue to **`04-11-06-synthetic-data.ipynb`** to learn how to generate diverse test scenarios automatically
- Or skip to **`04-11-07-tool-simulation.ipynb`** for LLM-powered tool simulation with `ToolSimulator`